# 12 — LLM suggestions for unresolved rows

This notebook suggests missing fields only for rows the canonical engine could not fully resolve. It now prioritizes deriving `DESCRIPTION` from the file's internal content (title, subject, heading, first meaningful line) before falling back to filename/path. It does not create final moves or rename files directly.


In [ ]:
from pathlib import Path
from datetime import datetime
import sys
import pandas as pd

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'notebooks').exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
POLICY_PATH = PROJECT_ROOT / 'policy' / 'SCH_fileserver_policy_v2_4.yaml'

from src.llm_suggestions import (
    SuggestionConfig,
    load_policy,
    merge_suggestions_into_candidates,
    normalize_suggestion_input,
    suggest_dataframe,
)

policy = load_policy(POLICY_PATH)
print('PROJECT_ROOT =', PROJECT_ROOT)


In [ ]:
def latest_output(prefix: str) -> Path:
    matches = sorted(OUTPUT_DIR.glob(f'{prefix}_*.parquet'))
    if not matches:
        matches = sorted(OUTPUT_DIR.glob(f'{prefix}_*.csv'))
    if not matches:
        raise FileNotFoundError(f'No output found for prefix: {prefix}')
    return matches[-1]

CANDIDATE_PATH = latest_output('canonical_candidates')
TEXT_PATH = latest_output('inventory_with_text') if list(OUTPUT_DIR.glob('inventory_with_text_*.parquet')) or list(OUTPUT_DIR.glob('inventory_with_text_*.csv')) else None
REVIEW_PATH = OUTPUT_DIR / 'review_snapshot_latest.parquet' if (OUTPUT_DIR / 'review_snapshot_latest.parquet').exists() else None

def read_any(path: Path | None) -> pd.DataFrame:
    if path is None:
        return pd.DataFrame()
    if path.suffix.lower() == '.parquet':
        return pd.read_parquet(path)
    return pd.read_csv(path)

candidates = read_any(CANDIDATE_PATH)
text_df = read_any(TEXT_PATH)
review_df = read_any(REVIEW_PATH)

if not text_df.empty:
    join_cols = [c for c in ['relative_path', 'filename', 'extracted_text', 'text_preview', 'text_status', 'text_error'] if c in text_df.columns]
    text_df = text_df[join_cols].drop_duplicates(subset=['relative_path'])
    candidates = candidates.merge(text_df, on='relative_path', how='left', suffixes=('', '_text'))

if not review_df.empty:
    join_cols = [c for c in ['relative_path', 'filename', 'rule_status', 'rule_reason'] if c in review_df.columns]
    review_df = review_df[join_cols].drop_duplicates(subset=['relative_path'])
    candidates = candidates.merge(review_df, on='relative_path', how='left', suffixes=('', '_review'))

candidates = normalize_suggestion_input(candidates)
print('Rows:', len(candidates))
display(candidates[[c for c in ['relative_path', 'filename', 'canonical_ready', 'unresolved_fields'] if c in candidates.columns]].head(10))


In [ ]:
USE_OLLAMA = False
OLLAMA_MODEL = 'llama3.1:8b'
MAX_ROWS = 50
TEXT_PREVIEW_CHARS = 8000
PREFER_CONTENT_FOR_DESCRIPTION = True

config = SuggestionConfig(
    use_ollama=USE_OLLAMA,
    ollama_model=OLLAMA_MODEL,
    max_rows=MAX_ROWS,
    include_text_preview_chars=TEXT_PREVIEW_CHARS,
    prefer_content_for_description=PREFER_CONTENT_FOR_DESCRIPTION,
)

suggestions = suggest_dataframe(candidates, policy, config=config)
print('Suggested rows:', len(suggestions))
display(suggestions[[c for c in ['relative_path', 'unresolved_fields', 'suggested_phase', 'suggested_doc_type', 'suggested_date', 'suggested_description', 'suggested_description_source', 'suggestion_confidence', 'suggestion_source'] if c in suggestions.columns]].head(20))


In [ ]:
merged = merge_suggestions_into_candidates(candidates, suggestions)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
suggestions_csv = OUTPUT_DIR / f'llm_suggestions_{timestamp}.csv'
suggestions_parquet = OUTPUT_DIR / f'llm_suggestions_{timestamp}.parquet'
merged_csv = OUTPUT_DIR / f'canonical_with_suggestions_{timestamp}.csv'
merged_parquet = OUTPUT_DIR / f'canonical_with_suggestions_{timestamp}.parquet'

suggestions.to_csv(suggestions_csv, index=False, encoding='utf-8-sig')
suggestions.to_parquet(suggestions_parquet, index=False)
merged.to_csv(merged_csv, index=False, encoding='utf-8-sig')
merged.to_parquet(merged_parquet, index=False)

print('Wrote:', suggestions_csv.name)
print('Wrote:', suggestions_parquet.name)
print('Wrote:', merged_csv.name)
print('Wrote:', merged_parquet.name)


In [ ]:
display(merged[[c for c in ['relative_path', 'filename', 'unresolved_fields', 'suggested_phase', 'suggested_doc_type', 'suggested_date', 'suggested_version', 'suggested_status', 'suggested_description', 'suggestion_confidence', 'suggestion_source'] if c in merged.columns]].head(30))

display(merged['suggestion_source'].fillna('missing').value_counts().rename_axis('suggestion_source').reset_index(name='count'))
